In [1]:
import importlib, pathlib, sys, time, urllib.request
from teehr.evaluation.spark_session_utils import create_spark_session

BRANCH = "improve-nwmd-preprocessing-efficiency"
MOD_DIR = pathlib.Path("/home/jovyan/nwmd_modules")
MOD_DIR.mkdir(parents=True, exist_ok=True)

RAW = ("https://raw.githubusercontent.com/RTIInternational/teehr-hub/"
       f"{BRANCH}/warehouse/remote/03_preprocessing/nwm_diagnostics/utils.py")

# cache-buster: raw.githubusercontent caches branch URLs for ~5 minutes
urllib.request.urlretrieve(f"{RAW}?t={time.time()}", MOD_DIR / "utils.py")

sys.path.insert(0, str(MOD_DIR))
import utils
importlib.reload(utils)  # picks up a re-download without a kernel restart
print("loaded", utils.__file__)


loaded /home/jovyan/nwmd_modules/utils.py


In [2]:
pod_template_path = utils.create_ondemand_pod_template()

Wrote alternate pod template to /home/jovyan/executor-pod-template-ondemand.yaml


In [15]:
# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=64,
#     executor_memory="16g",
#     executor_cores=2,
#     aws_profile="default",
#     pod_template_path=pod_template_path,
#     update_configs={
#         "spark.sql.shuffle.partitions": 1024,
#         "spark.sql.adaptive.coalescePartitions.enabled": "false",
#         "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
#         "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
#         "spark.executor.memoryOverhead": "4g",
#     }
# )

spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [7]:
spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_test PURGE")

DataFrame[]

In [11]:
configurations = [
    # {
    #     "configurations": ["nwm30_medium_range"],
    #     "forecast_lead_time_bin_hours": 24,
    #     "start_reference_time": "2025-10-01T00:00",
    #     "end_reference_time": "2026-10-01T00:00"
    # },
    {
        "configurations": ["nwm30_short_range"],
        "forecast_lead_time_bin_hours": 6,
        "start_reference_time": "2025-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00"
    }
]

In [12]:
for config in configurations:
    print(config)
    utils.generate_nwmd_metrics(spark, config)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.
INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.


{'configurations': ['nwm30_short_range'], 'forecast_lead_time_bin_hours': 6, 'start_reference_time': '2025-10-01T00:00', 'end_reference_time': '2026-10-01T00:00'}
Unique fields for grouping: ['reference_time', 'primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member']


INFO:teehr.evaluation.dataframe_base:Setting filter id like 'usgs-%'.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.isin: 'in'>, value=['nwm30_short_range']), TableFilter(column='reference_time', operator=<FilterOperators.gte: '>='>, value='2025-10-01T00:00'), TableFilter(column='reference_time', operator=<FilterOperators.lt: '<'>,

Number of location_ids: 10


INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Setting order_by ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg'].
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.dataframe_base:Writing to table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_test.
INFO:teehr.evaluation.re

203.648935 s


In [ ]:
spark.sql("USE iceberg.teehr")

In [30]:
spark.sql("""
SELECT *
FROM nwmd_metrics_by_location_test 
LIMIT 10
""").show()

+-------------------+---------------------+------------------+---------+--------------------+------+-------+----------------------+---------+----------+-----+------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+---------------------------+-------------------+-------------------------+----------------------+--------------------+------------------------+------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------------------+--------------------------------------+------------------------------------+------------------------------------+------------------------+------------------------+------------------------------+------------------------------+---------------------------------+---------------------------------+--------------------+-------------------

In [32]:

spark.sql("""
SELECT forecast_lead_time_bin, max(count) 
FROM nwmd_metrics_by_location_test 
WHERE configuration_name = 'nwm30_short_range'
GROUP BY forecast_lead_time_bin


""").show()


+----------------------+----------+
|forecast_lead_time_bin|max(count)|
+----------------------+----------+
|             PT0S_PT6H|      2208|
|            PT6H_PT12H|      2208|
|           PT12H_PT18H|      2208|
|          PT18H_P1DT0H|      2208|
+----------------------+----------+



In [14]:
spark.stop()